In [1]:
# %%
# ============================================================
# CELDA 1 — Imports y configuración
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW  = os.path.join(BASE, 'data', 'raw')

print(f"Base del proyecto: {BASE}")
print("Librerías cargadas correctamente")

# %%
# ============================================================
# CELDA 2 — Carga y combinación de las 9 ciudades
# ============================================================

CIUDADES = {
    'barcelona': ['jun2025', 'sep2025'],
    'euskadi':   ['jun2025', 'sep2025'],
    'girona':    ['jun2025', 'sep2025', 'dic2025'],
    'madrid':    ['jun2025', 'sep2025'],
    'malaga':    ['jun2025', 'sep2025'],
    'mallorca':  ['jun2025', 'sep2025'],
    'menorca':   ['jun2025', 'sep2025'],
    'sevilla':   ['jun2025', 'sep2025'],
    'valencia':  ['jun2025', 'sep2025'],
}

def cargar(ruta, ciudad, fecha):
    df = pd.read_csv(ruta, compression='gzip', low_memory=False)
    df['ciudad']       = ciudad
    df['fecha_scrape'] = fecha
    return df

dfs = []
for ciudad, periodos in CIUDADES.items():
    for periodo in periodos:
        ruta = os.path.join(RAW, ciudad, periodo, 'listings.csv.gz')
        if os.path.exists(ruta):
            df_tmp = cargar(ruta, ciudad, periodo)
            dfs.append(df_tmp)
            print(f"✅ {ciudad:12} | {periodo} | {len(df_tmp):6,} anuncios")
        else:
            print(f"❌ {ciudad:12} | {periodo} | NO ENCONTRADO")

df = pd.concat(dfs, ignore_index=True)

print(f"\nTOTAL: {len(df):,} anuncios | {df.shape[1]} columnas")
print(f"\n--- Anuncios por ciudad y fecha ---")
print(df.groupby(['ciudad', 'fecha_scrape']).size().reset_index(name='anuncios').to_string(index=False))

# Verificamos columnas clave
cols_clave = ['price', 'estimated_occupancy_l365d', 'estimated_revenue_l365d',
              'neighbourhood_cleansed', 'room_type', 'accommodates', 'bedrooms']
print(f"\n--- Columnas clave presentes ---")
for col in cols_clave:
    print(f"   {'✅' if col in df.columns else '❌'} {col}")

# %%
# ============================================================
# CELDA 3 — Selección de variables y limpieza
# ============================================================

COLUMNAS = [
    'id', 'ciudad', 'fecha_scrape',
    'neighbourhood_cleansed',
    'latitude', 'longitude',
    'property_type', 'room_type', 'accommodates',
    'bedrooms', 'beds', 'bathrooms_text',
    'price',
    'minimum_nights',
    'availability_30', 'availability_60',
    'availability_90', 'availability_365',
    'number_of_reviews', 'number_of_reviews_ltm',
    'review_scores_rating', 'review_scores_cleanliness',
    'review_scores_location', 'review_scores_value',
    'reviews_per_month',
    'host_is_superhost', 'host_response_rate',
    'host_acceptance_rate', 'host_listings_count',
    'instant_bookable',
    'estimated_occupancy_l365d',
    'estimated_revenue_l365d',
]

df_sel = df[COLUMNAS].copy()
print(f"Antes de limpieza: {len(df_sel):,} anuncios")

# 1. Eliminamos filas sin precio
df_clean = df_sel.dropna(subset=['price']).copy()
print(f"Tras eliminar sin precio: {len(df_clean):,}")

# 2. Limpiamos precio
df_clean['price'] = (
    df_clean['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# 3. Filtramos precios imposibles
df_clean = df_clean[(df_clean['price'] > 0) & (df_clean['price'] <= 5000)]
print(f"Tras filtrar precios: {len(df_clean):,}")

# 4. Limpiamos bathrooms_text → número
df_clean['bathrooms'] = (
    df_clean['bathrooms_text']
    .str.extract(r'(\d+\.?\d*)')
    .astype(float)
)
df_clean = df_clean.drop(columns=['bathrooms_text'])

# 5. Limpiamos booleanos
df_clean['host_is_superhost'] = df_clean['host_is_superhost'].map({'t': 1, 'f': 0})
df_clean['instant_bookable']  = df_clean['instant_bookable'].map({'t': 1, 'f': 0})

# 6. Limpiamos porcentajes
for col in ['host_response_rate', 'host_acceptance_rate']:
    df_clean[col] = (
        df_clean[col]
        .str.replace('%', '', regex=False)
        .astype(float)
    )

# 7. Ocupación corregida: noches ocupadas / 365
df_clean['ocupacion'] = df_clean['estimated_occupancy_l365d'] / 365

print(f"\n--- Distribución por ciudad tras limpieza ---")
print(df_clean.groupby(['ciudad', 'fecha_scrape']).size()
      .reset_index(name='anuncios').to_string(index=False))

print(f"\n--- Estadísticas precio por ciudad ---")
print(df_clean.groupby('ciudad')['price'].describe().round(2))

print(f"\n--- Estadísticas ocupación por ciudad ---")
print(df_clean.groupby('ciudad')['ocupacion'].describe().round(3))

# %%
# ============================================================
# CELDA 4 — Diagnóstico variable ocupación
# ============================================================

print("--- estimated_occupancy_l365d (muestra) ---")
print(df_clean['estimated_occupancy_l365d'].describe())

print(f"\n--- Distribución por rangos ---")
rangos = pd.cut(df_clean['ocupacion'],
                bins=[0, 0.25, 0.50, 0.75, 1.0],
                labels=['0-25%', '25-50%', '50-75%', '75-100%'])
print(rangos.value_counts().sort_index())

print(f"\n--- Ingresos anuales estimados por ciudad (€) ---")
print(df_clean.groupby('ciudad')['estimated_revenue_l365d'].describe().round(0))

# %%
# ============================================================
# CELDA 5 — Guardado dataset limpio
# ============================================================

PROCESSED = os.path.join(BASE, 'data', 'processed')
os.makedirs(PROCESSED, exist_ok=True)

ruta_salida = os.path.join(PROCESSED, 'listings_clean.csv')
df_clean.to_csv(ruta_salida, index=False)

tam = os.path.getsize(ruta_salida) / (1024*1024)
print(f"✅ Dataset guardado en: {ruta_salida}")
print(f"   Tamaño: {tam:.1f} MB")
print(f"   Filas:  {len(df_clean):,}")
print(f"   Columnas: {df_clean.shape[1]}")
print(f"\n--- Columnas finales ---")
for col in df_clean.columns:
    print(f"   → {col}")

Base del proyecto: /Users/ivannavarrosuero/tfm_vut
Librerías cargadas correctamente
✅ barcelona    | jun2025 | 18,927 anuncios
✅ barcelona    | sep2025 | 19,410 anuncios
✅ euskadi      | jun2025 |  7,151 anuncios
✅ euskadi      | sep2025 |  6,334 anuncios
✅ girona       | jun2025 | 22,456 anuncios
✅ girona       | sep2025 | 17,069 anuncios
✅ girona       | dic2025 | 17,069 anuncios
✅ madrid       | jun2025 | 26,004 anuncios
✅ madrid       | sep2025 | 25,000 anuncios
✅ malaga       | jun2025 |  9,710 anuncios
✅ malaga       | sep2025 |  9,714 anuncios
✅ mallorca     | jun2025 | 17,113 anuncios
✅ mallorca     | sep2025 | 15,034 anuncios
✅ menorca      | jun2025 |  3,951 anuncios
✅ menorca      | sep2025 |  3,679 anuncios
✅ sevilla      | jun2025 |  8,323 anuncios
✅ sevilla      | sep2025 |  8,215 anuncios
✅ valencia     | jun2025 |  9,009 anuncios
✅ valencia     | sep2025 |  7,844 anuncios

TOTAL: 252,012 anuncios | 87 columnas

--- Anuncios por ciudad y fecha ---
   ciudad fecha_scrape 